In [17]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from dython.nominal import associations
import copy
import math

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [7]:
#let's one-hot-encode for procedure, type contract, central purchasing, eu_fund, framework or dynamic purchasing, 
base_n_encoder_cols = ["cpv_code", "country", "language"]
one_hot_columns = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "contracting authority type", "contracting authority activity"]
numerical_columns = ["duration", "nb_tenders_received", "nb_tenders_received_sme", "award_contract/val_total: 0"]
text_embedding_columns = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]
categorical_columns_original = base_n_encoder_cols + one_hot_columns

In [11]:
df_total_clean = pd.read_pickle("new_data/df_merged_dataset_clean")

#path = r"C:\Users\bbinnend\OneDrive\Thesis\open_data_can_2020_10.07.23.csv"
#df_total = pd.read_csv(path)

In [12]:
numerical_columns = ["AWARD_VALUE_EURO_FIN_1","LOTS_NUMBER", "CRIT_PRICE_WEIGHT", "NUMBER_OFFERS", "NUMBER_TENDERS_SME"]

df_total_1 = df_total_clean.loc[df_total_clean["CRIT_PRICE_WEIGHT"] < 100]
df_total_2 = df_total_clean.loc[df_total_clean["NUMBER_TENDERS_SME"] != 0]
#df_total_3 = df_total_clean.loc[(df_total_clean["AWARD_VALUE_EURO_FIN_1"] > 5000 & df_total_clean["AWARD_VALUE_EURO_FIN_1"] < 800000)]

In [ ]:
correlation_matrix

In [ ]:
numerical_columns = ["AWARD_VALUE_EURO_FIN_1","LOTS_NUMBER", "CRIT_PRICE_WEIGHT", "NUMBER_OFFERS", "NUMBER_TENDERS_SME"]
#define correlation matrix
correlation_matrix = df_total_1[numerical_columns].corr()
plt.figure(figsize=(10, 8))  # You can adjust the figure size as needed
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Plot')
plt.show()


In [11]:
df_train_original = pd.read_pickle("Data/df_preprocessed_no_encoding_test.pickle").reset_index(drop = True).drop(columns = ["original index", "date_conclusion_contract"])
df_test_original = pd.read_pickle("Data/df_preprocessed_no_encoding_test.pickle").reset_index(drop = True).drop(columns = ["original index", "date_conclusion_contract"])
df_total_no_encoding = pd.concat([df_train_original, df_test_original])

In [8]:
drop_cols = ["ID_NOTICE_CAN", "short description"]
df_total = df_total.drop(columns = drop_cols)

In [ ]:
df_total.head()

In [18]:
def filter_outliers(df, upper, lower, target_col = "AWARD_VALUE_EURO_FIN_1"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers, with high amount = {}, and low amount = {}".format(len(outlier_indices), high_amount, low_amount))
    df = df.drop(labels = outlier_indices, axis = 0)
    return df

In [ ]:
df_total_clean = df_total_clean.reset_index(drop=True)
df_total = filter_outliers(df_total_clean, upper=0.90, lower=0.20)

In [24]:
df_total = df_total.drop(columns=["ID_NOTICE_CAN", "short description", ""])

In [ ]:
for col in df_total.columns:
    print(col, ": ", len(df_total[col].value_counts()))

In [ ]:
df_total.head()

In [ ]:
#create a correlation plot between the categorical columns
df_complete_corr = associations(df_total, filename="Figures/data exploration results/corr_plot_cat_var.png", figsize=(12,12))

In [ ]:
df_total.head()

In [42]:
results_language_model = {"training loss": [53043462668.288002, 54128852074.496002, 52867306094.592003, 52855406854.143997, 54155850809.344002, 53962199793.664001, 53760822870.015999],
                          "validation loss": [50737639424.000000, 50729619456.000000, 50722869248.000000, 50717372416.000000, 50713128960.000000, 50710142976.000000, 50708410368.000000],
                          "mse": [50737639424.000000, 50729623552.000000, 50722869248.000000, 50717372416.000000, 50713128960.000000, 50710138880.000000, 50708410368.000000],
                          "mae": [130941.226562, 130910.601562, 130884.796875, 130863.804688, 130847.585938, 130836.171875, 130829.546875]}

In [34]:
#with open("new_data/training_results_language_model.pickle", "wb") as f:
#    pickle.dump(results_language_model, f)

In [ ]:
import math
math.sqrt(3.798)

In [36]:
import pickle
with open("history_combined_model", "rb") as f:
    language_model_results = pickle.load(f)

In [ ]:
language_model_results

In [ ]:
del language_model_results["loss"]
language_model_results

In [40]:
def plot_metrics(ledger, save_path = None):
  #model results is a dict from a keras object
  model_results = ledger
  
  plt.figure(figsize=(10, 10))

  # Create the top row of subplots
  ax1 = plt.subplot(3, 1, 1)
  ax2 = plt.subplot(3, 1, 2)
  ax3 = plt.subplot(3, 1, 3)
  axes = [ax1, ax2, ax3]

  line_styles = ['-', '-.']
  markers = ['o', 's']
  for i, key in enumerate(list(model_results.keys())[:int(0.5*len(model_results.keys()))]):
    loss_train = model_results[key]
    loss_val   = model_results["val_" + key]
    epochs = range(0,len(loss_train))

    axes[i].plot(epochs, loss_train, "g", label = "Training {}".format(key), color='grey', linestyle=line_styles[0], marker=markers[0])
    axes[i].plot(epochs, loss_val, "b", label = "Validation {}".format("val_" + key), color='grey', linestyle=line_styles[1], marker=markers[1])
    axes[i].title.set_text("Training and validation {}".format(key))
    axes[i].set_xlabel("Epochs")
    axes[i].set_ylabel("{}".format(key))
    axes[i].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='png')

In [ ]:
plot_metrics(results_language_model)